## Part1 Baseline Discrete Local Search (Hill‑Climber)
### Student: Beverly, Echo
### Student ID: 202316801316,202316801317

### Common utility functions
Below are shared helper functions reused by all group members:
1. `calculate_total_resource()`: compute total hardware resource consumption for a binary solution
2. `is_feasible()`: check whether a solution respects all hardware capacity bounds
3. `fitness()`: objective function with **penalty‑based constraint handling**.
If solution violates resource constraints, heavy penalty is subtracted from utility value to discourage infeasible solutions.

In [4]:
import numpy
import scipy
print(numpy.__version__, scipy.__version__)

1.26.4 1.13.1


In [6]:
import numpy as np
import matplotlib.pyplot as plt

# ========= Shared group functions (consistent with team‑member interface) =========
def load_instance(file_path):
    
    with open(file_path, 'r') as f:
        tokens = f.read().split()

    if not tokens:
        raise ValueError(f"Dataset file at '{file_path}' is empty or invalid.")

    n = int(tokens[0])
    m = int(tokens[1])
    idx = 2

    v = np.array([float(x) for x in tokens[idx: idx + n]])
    idx += n

    W = np.zeros((m, n))
    for i in range(m):
        W[i, :] = [float(x) for x in tokens[idx: idx + n]]
        idx += n

    b = np.array([float(x) for x in tokens[idx: idx + m]])
    return n, m, v, W, b

def is_feasible(sol, W, b):
    "Check if the solution is feasible: sol is a 0/1 array"
    cons = W @ sol
    return np.all(cons <= b + 1e-9)

def evaluate_fitness(sol, v, W, b):
    "Basic fitness: Feasible solutions = total value; Infeasible = 0"
    if is_feasible(sol, W, b):
        return float(v @ sol)
    else:
        return 0.0


# ========== Variant B Core: Repair Recovery Strategy  ==========
def repair_solution(sol, W, b, v):

    sol = sol.copy().astype(int)
    n = len(sol)
    # value density
    density = np.zeros(n)
    mask = sol == 1
    density[mask] = v[mask] / (W[:,mask].sum(axis=0) + 1e-9)
    # Retrieve the index of the currently selected item, sort them in ascending order of density
    selected_idx = np.where(sol == 1)[0]
    selected_idx = selected_idx[np.argsort(density[selected_idx])]

    for idx in selected_idx:
        if is_feasible(sol, W, b):
            break
        sol[idx] = 0
    return sol


# ==================== Variant‑A Penalty‑based constraint handling (Part‑1 Hill‑Climber) ====================
def fitness_penalty(sol, v, W, b, penalty_weight=10000):
    
    total_val = np.sum(v * sol)
    if is_feasible(sol, W, b):
        return total_val
    overshoot = np.maximum((W @ sol) - b, 0.0)
    return total_val - penalty_weight * np.sum(overshoot)